# Cloud Cover Analysis for Mission Planning

Cloud cover is a primary constraint for airborne remote sensing campaigns.
HyPlan's `clouds` module uses **Google Earth Engine** to retrieve historical
MODIS cloud fraction data and simulate realistic mission visit schedules.

We cover:

1. Google Earth Engine setup and authentication
2. Retrieving MODIS cloud fraction for study areas
3. Simulating mission visits with cloud, rest, and weekend constraints
4. Visualizing cloud fraction heatmaps with visit markers
5. Interpreting results for campaign planning

> **Prerequisite:** This notebook requires a Google Earth Engine account.
> Sign up at [earthengine.google.com](https://earthengine.google.com/)
> and authenticate before running.

In [ ]:
import ee

# Authenticate and initialize Earth Engine
# If running for the first time, this will open a browser for authentication.
try:
    ee.Initialize()
    print("Earth Engine initialized successfully.")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine authenticated and initialized.")

In [ ]:
from hyplan.clouds import (
    create_cloud_data_array_with_limit,
    simulate_visits,
    plot_yearly_cloud_fraction_heatmaps_with_visits,
)

## 1. Study Area Definition

Cloud data is retrieved for polygons defined in a GeoJSON file. Each
polygon represents a study area (flight box) that needs to be imaged.
The GeoJSON must have a `Name` column identifying each polygon.

We use a sample file from the examples directory.

In [ ]:
import geopandas as gpd

polygon_file = "../notebooks/exampledata/wdts.geojson"

# Preview the study areas
gdf = gpd.read_file(polygon_file)
print(f"Study areas: {len(gdf)} polygons")
print(gdf[["Name"]].to_string(index=False))

gdf.plot(figsize=(8, 5), alpha=0.4, edgecolor="black")

## 2. Retrieve MODIS Cloud Fraction

`create_cloud_data_array_with_limit` queries Google Earth Engine for
daily MODIS cloud fraction over each polygon across multiple years.
This gives the historical cloud climatology needed to estimate how
many clear days are available during a campaign window.

Parameters:
- `polygon_file`: GeoJSON with study-area polygons
- `year_start`, `year_stop`: Range of years to query
- `day_start`, `day_stop`: Day-of-year window (e.g., 1-75 for Jan-Mar)

In [ ]:
# Retrieve cloud data for January through mid-March, 2003-2006
cloud_data_df = create_cloud_data_array_with_limit(
    polygon_file=polygon_file,
    year_start=2003,
    year_stop=2022,
    day_start=1,
    day_stop=75,
)

print(f"Cloud data shape: {cloud_data_df.shape}")
print(f"Columns: {list(cloud_data_df.columns)}")
cloud_data_df.head(10)

## 3. Simulate Mission Visits

`simulate_visits` takes the cloud fraction data and simulates a
realistic mission schedule. On each day, it checks whether each
study area is below the cloud fraction threshold. If so, the area
is "visited" (imaged). The simulation respects:

- **Cloud fraction threshold** — maximum cloudiness for a flyable day
- **Rest day schedule** — mandatory rest after N consecutive flight days
- **Weekend exclusion** — optionally skip weekends

In [ ]:
# Simulate visits for February-March window
day_start = 30
day_stop = 74

visit_days_df, visit_tracker, rest_days = simulate_visits(
    cloud_data_df,
    day_start=day_start,
    day_stop=day_stop,
    year_start=2003,
    year_stop=2022,
    cloud_fraction_threshold=0.25,  # Max 25% cloud cover
    rest_day_threshold=5,           # Rest after 2 consecutive flight days
    exclude_weekends=True,          # No flights on weekends
    debug=True,
)

print(f"\nVisit results:")
print(f"  Shape: {visit_days_df.shape}")
visit_days_df.head()

## 4. Visualize Cloud Fraction Heatmaps

The heatmap shows daily cloud fraction for each study area across the
campaign window. Visit days are marked, along with rest days and
weekends, giving a complete picture of mission scheduling constraints.

In [ ]:
plot_yearly_cloud_fraction_heatmaps_with_visits(
    cloud_data_df,
    visit_tracker,
    rest_days,
    cloud_fraction_threshold=0.25,
    exclude_weekends=True,
    day_start=day_start,
    day_stop=day_stop,
)

## 5. Campaign Planning Insights

Use the simulation results to answer key planning questions:

- How many flyable days per study area?
- What cloud threshold gives adequate coverage?
- How long should the campaign window be?

In [ ]:
import pandas as pd
import numpy as np

# --- Question 1: How many flyable days per study area? ---
# Use the 25% threshold simulation from Section 3

polygons = sorted(cloud_data_df["polygon_id"].unique())
years = list(range(2003, 2007))

print("Flyable days per study area (25% cloud threshold):\n")
print(f"{'Area':<12s}", end="")
for y in years:
    print(f"  {y}", end="")
print("   Mean")
print("-" * 48)

for poly in polygons:
    counts = []
    for y in years:
        n = len(visit_tracker.get(y, {}).get(poly, []))
        counts.append(n)
    print(f"{poly:<12s}", end="")
    for c in counts:
        print(f"  {c:4d}", end="")
    print(f"   {np.mean(counts):4.1f}")

# Total flight days per year (any area visited)
print(f"\n{'TOTAL':<12s}", end="")
for y in years:
    all_days = set()
    for poly in polygons:
        all_days.update(visit_tracker.get(y, {}).get(poly, []))
    print(f"  {len(all_days):4d}", end="")
print()

In [ ]:
# --- Question 2: What cloud threshold gives adequate coverage? ---
# Compare how many areas get visited at least once per year across thresholds

thresholds = [0.10, 0.15, 0.20, 0.25, 0.35, 0.50]

rows = []
for thresh in thresholds:
    _, vt, _ = simulate_visits(
        cloud_data_df,
        day_start=day_start,
        day_stop=day_stop,
        year_start=2003,
        year_stop=2006,
        cloud_fraction_threshold=thresh,
        rest_day_threshold=2,
        exclude_weekends=True,
        debug=False,
    )
    for y in years:
        visits_per_area = {p: len(vt.get(y, {}).get(p, [])) for p in polygons}
        rows.append({
            "threshold": thresh,
            "year": y,
            "areas_visited": sum(1 for v in visits_per_area.values() if v > 0),
            "total_visits": sum(visits_per_area.values()),
            "min_visits": min(visits_per_area.values()),
        })

threshold_df = pd.DataFrame(rows)
summary = threshold_df.groupby("threshold").agg(
    avg_areas_visited=("areas_visited", "mean"),
    avg_total_visits=("total_visits", "mean"),
    avg_min_visits=("min_visits", "mean"),
).reset_index()

print("Impact of cloud fraction threshold (averaged over 2003-2006):\n")
print(f"{'Threshold':>10s}  {'Areas Visited':>14s}  {'Total Visits':>13s}  {'Min per Area':>13s}")
print("-" * 56)
for _, row in summary.iterrows():
    print(f"{row['threshold']:>10.0%}  {row['avg_areas_visited']:>14.1f}  {row['avg_total_visits']:>13.1f}  {row['avg_min_visits']:>13.1f}")

In [ ]:
# --- Question 3: How long should the campaign window be? ---
# Simulate different window lengths and see how many areas get covered

window_lengths = [14, 21, 30, 45, 60]
# Center windows around DOY 52 (late Feb)
center_doy = 52

rows = []
for wlen in window_lengths:
    ds = center_doy - wlen // 2
    de = ds + wlen - 1
    _, vt, _ = simulate_visits(
        cloud_data_df,
        day_start=ds,
        day_stop=de,
        year_start=2003,
        year_stop=2006,
        cloud_fraction_threshold=0.25,
        rest_day_threshold=2,
        exclude_weekends=True,
        debug=False,
    )
    for y in years:
        visits_per_area = {p: len(vt.get(y, {}).get(p, [])) for p in polygons}
        all_flight_days = set()
        for p in polygons:
            all_flight_days.update(vt.get(y, {}).get(p, []))
        rows.append({
            "window_days": wlen,
            "year": y,
            "areas_covered": sum(1 for v in visits_per_area.values() if v > 0),
            "total_visits": sum(visits_per_area.values()),
            "flight_days": len(all_flight_days),
        })

window_df = pd.DataFrame(rows)
wsummary = window_df.groupby("window_days").agg(
    avg_areas=("areas_covered", "mean"),
    avg_visits=("total_visits", "mean"),
    avg_flight_days=("flight_days", "mean"),
).reset_index()

print("Campaign window length vs. coverage (25% threshold, centered on late Feb):\n")
print(f"{'Window (days)':>14s}  {'Areas Covered':>14s}  {'Total Visits':>13s}  {'Flight Days':>12s}")
print("-" * 58)
for _, row in wsummary.iterrows():
    print(f"{row['window_days']:>14.0f}  {row['avg_areas']:>14.1f}  {row['avg_visits']:>13.1f}  {row['avg_flight_days']:>12.1f}")